In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

import os
import string
import itertools
from collections import Counter

randseed_date = 20260721
randseed_date_previous = 20260630
np.random.seed(randseed_date_previous) # keep same random seed as last experiment to aid in sequencer re-usability

## setup the 384 well format and the picklist of strains

In [3]:
from Bio import SeqIO
import re

# Read the fasta file and extract strain numbers
fasta_file = "/home/rl/scripts/karl/merge_consensus_sequences/collapse_naive_updated3_15diff/corroborated_db_filtered_min5.fasta"
strain_numbers = []

for record in SeqIO.parse(fasta_file, "fasta"):
    # Extract the part before the first underscore
    strain = record.id.split('_')[0]
    if strain not in strain_numbers:
        strain_numbers.append(strain)

# Sort alphabetically with natural sorting (A2 before A10)
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

strain_numbers.sort(key=natural_sort_key)
print(strain_numbers)

['A5', 'A6', 'A11', 'A19', 'A20', 'B4', 'B7', 'B12', 'B15', 'B24', 'C3', 'C8', 'C11', 'C12', 'C14', 'C15', 'C19', 'C20', 'D1', 'D3', 'D6', 'D7', 'D11', 'D13', 'D15', 'D20', 'E3', 'E14', 'E24', 'F2', 'F7', 'F13', 'F16', 'F17', 'F19', 'G5', 'G8', 'G17', 'H20', 'I19', 'J2', 'J5', 'J18', 'J22', 'J24', 'K2', 'K3', 'K6', 'K8', 'K10', 'K11', 'K15', 'K17', 'L6', 'L11', 'L19', 'L23', 'L24', 'M12', 'M14', 'M15', 'M23', 'N3', 'N10', 'N13', 'N14', 'N15', 'N19', 'N21', 'N23', 'N24', 'O5', 'O7', 'O10', 'O17', 'O18', 'O21', 'O24', 'P1', 'P8', 'P11', 'P13', 'P14', 'P16', 'P17', 'P20', 'P23']


In [4]:
pick_list = set(strain_numbers)
#pick_list = {'F17', 'O9', 'F5', 'G3', 'E24', 'C14', 'M19', 'E2', 'H11', 'E10', 'D23', 'E13', 'O11', 'G6', 'E4', 'A2', 'O13', 'I3', 'A12', 'G18', 'G4', 'O3', 'G9', 'A21', 'C22', 'E20', 'O21', 'A6', 'P15', 'A14', 'F15', 'G13', 'G12', 'P17', 'E15', 'M9', 'J5', 'E9', 'G1', 'J13', 'M17', 'E14', 'E7', 'E5', 'I9', 'J9', 'M11', 'H17', 'E6', 'P21', 'A1', 'G2', 'E12', 'O7', 'E16', 'A11', 'F23', 'P11', 'A15', 'E21', 'C24', 'O19', 'E22', 'G5', 'J1', 'A8', 'H23', 'G20', 'G8', 'C20', 'I11', 'G19', 'E8', 'G15', 'O17', 'G7', 'H15', 'G17', 'H19', 'E18', 'F19', 'M21', 'F7', 'G21', 'O15', 'M13', 'P19', 'G22', 'G11', 'F13'}
no_growth_strains = {None}#{'P23', 'H5', 'F21', 'G23', 'A10', 'J3', 'H7', 'J11', 'G10', 'G14'}
#pick list from plate reader data in the strain_check folder

n_rows, n_cols = 16, 24
wells384 = np.array([r + str(c) for r in string.ascii_uppercase[:n_rows] for c in range(1, n_cols + 1)])
wells384_2d = wells384.reshape(n_rows, n_cols)
# print(wells384)

# Step 1 — PCR primer layout (frame conditions)

Max out the PCR primer layout. There are **96 forward** and **96 reverse** barcodes. The only invalid combinations are where the forward and reverse barcode have the **same index** (`i == j`). Every other pairing is valid, giving `96 * 96 - 96 = 9120` combinations.

We lay each combination out into 384-well destination plates (`dest_row`, `dest_col`, `dest_plate`), randomize the placement with `np.random`, and emit the corresponding Echo transfer dataframe + CSV. `9120 / 308 = 29.6`, so this fills **30 plates** (the last one ~1/2 full).

In [5]:
# --- Source plate primer layout (same checkerboard scheme as the drug experiment) ---
n_rows_nonedge, n_cols_nonedge = n_rows - 2, n_cols - 2  # 14x22 = 308 wells for destination plates
fprimers_full = wells384_2d[::2, ::2].flatten()    # 96 forward primer source wells (even rows/cols)
rprimers_full = wells384_2d[1::2, 1::2].flatten()  # 96 reverse primer source wells (odd rows/cols)
n_fwd, n_rev = len(fprimers_full), len(rprimers_full)
print(f"forward primers: {n_fwd}, reverse primers: {n_rev}")

# --- every valid forward x reverse combination (exclude identical index pairs) ---
combos = []
for i in range(n_fwd):
    for j in range(n_rev):
        if i != j:
            combos.append((i, j))
combos = np.array(combos)
print(f"total valid combos: {len(combos)} (expected {n_fwd * n_rev - n_fwd})")

# randomize plate placement
np.random.shuffle(combos)

# --- assign each combo to a destination row / col / plate, filling plates row-major ---

wells_per_plate = n_rows_nonedge * n_cols_nonedge  # 308
n_plates = int(np.ceil(len(combos) / wells_per_plate))
print(f"plates needed: {n_plates}")

records = []
for idx, (fi, rj) in enumerate(combos):
    plate  = idx // wells_per_plate + 1
    within = idx %  wells_per_plate 
    row_i  = (within // n_cols_nonedge) + 1  # skip first row (A)
    col_i  = (within %  n_cols_nonedge  ) + 1  # skip first col (1)
    dest_row = string.ascii_uppercase[row_i]
    dest_col = col_i + 1
    records.append({
        'dest_plate': plate,
        'dest_row': dest_row,
        'dest_col': dest_col,
        'dest_well': f"{dest_row}{dest_col}",
        'fwd_idx': int(fi),
        'rev_idx': int(rj),
        'fwd_source_well': fprimers_full[fi],
        'rev_source_well': rprimers_full[rj],
    })
df_layout = pd.DataFrame(records)

# --- sanity checks ---
assert (df_layout['fwd_idx'] == df_layout['rev_idx']).sum() == 0, "found identical-index pairs"
assert df_layout[['fwd_idx', 'rev_idx']].drop_duplicates().shape[0] == len(df_layout), "duplicate combos"
assert set(df_layout['fwd_source_well']).isdisjoint(set(df_layout['rev_source_well'])), "fwd/rev source collision"
print(f"last plate fill: {(df_layout['dest_plate'] == n_plates).sum()} / {wells_per_plate}")
display(df_layout)

forward primers: 96, reverse primers: 96
total valid combos: 9120 (expected 9120)
plates needed: 30
last plate fill: 188 / 308


,dest_plate,dest_row,dest_col,dest_well,fwd_idx,rev_idx,fwd_source_well,rev_source_well
0,1,B,2,B2,31,59,E15,J24
1,1,B,3,B3,86,24,O5,F2
2,1,B,4,B4,77,67,M11,L16
3,1,B,5,B5,39,65,G7,L12
4,1,B,6,B6,40,5,G9,B12
...,...,...,...,...,...,...,...,...
9115,30,J,9,J9,48,7,I1,B16
9116,30,J,10,J10,70,13,K21,D4
9117,30,J,11,J11,61,49,K3,J4
9118,30,J,12,J12,55,24,I15,F2


In [6]:
# --- Echo transfer df: two rows per destination well (forward primer + reverse primer) ---
vol = 250  # nL per primer transfer (matches the drug experiment)

echo_rows = []
for r in df_layout.itertuples():
    echo_rows.append({'Source Well': r.fwd_source_well, 'Destination Plate Name': r.dest_plate,
                      'Destination Well': r.dest_well, 'Transfer Volume': vol})
    echo_rows.append({'Source Well': r.rev_source_well, 'Destination Plate Name': r.dest_plate,
                      'Destination Well': r.dest_well, 'Transfer Volume': vol})
df_echo = pd.DataFrame(echo_rows)
print(f"echo rows: {len(df_echo)} (2 x {len(df_layout)} = {2 * len(df_layout)})")

# full layout (with primer indices/source wells) and the Echo-ready transfer file
df_layout.to_csv(f"primer_layout_{randseed_date}.csv", index=False)
# df_echo.to_csv(f"echo_primers_{randseed_date}.csv", index=False)
display(df_layout)
display(df_echo)

echo rows: 18240 (2 x 9120 = 18240)


,dest_plate,dest_row,dest_col,dest_well,fwd_idx,rev_idx,fwd_source_well,rev_source_well
0,1,B,2,B2,31,59,E15,J24
1,1,B,3,B3,86,24,O5,F2
2,1,B,4,B4,77,67,M11,L16
3,1,B,5,B5,39,65,G7,L12
4,1,B,6,B6,40,5,G9,B12
...,...,...,...,...,...,...,...,...
9115,30,J,9,J9,48,7,I1,B16
9116,30,J,10,J10,70,13,K21,D4
9117,30,J,11,J11,61,49,K3,J4
9118,30,J,12,J12,55,24,I15,F2


,Source Well,Destination Plate Name,Destination Well,Transfer Volume
0,E15,1,B2,250
1,J24,1,B2,250
2,O5,1,B3,250
3,F2,1,B3,250
4,M11,1,B4,250
...,...,...,...,...
18235,J4,30,J11,250
18236,I15,30,J12,250
18237,F2,30,J12,250
18238,I3,30,J13,250


In [7]:
#sum how much will be shot from each well with a groupby on df_strain_echo
primer_shots = df_echo.groupby('Source Well')['Transfer Volume'].sum()
display(primer_shots)

Source Well
A1     23750
A11    23750
A13    23750
A15    23750
A17    23750
       ...  
P22    23750
P24    23750
P4     23750
P6     23750
P8     23750
Name: Transfer Volume, Length: 192, dtype: int64

In [8]:

out_dir = "primer_shooting"
os.makedirs(out_dir, exist_ok=True)

for i in range(1, n_plates + 1):
    # print(f"Processing plate {i+1}")
    df_i = df_echo[df_echo['Destination Plate Name'] == i]
    path = os.path.join(out_dir, f"echo_primers_{randseed_date}_part_{i:02d}.csv")
    df_i.to_csv(path, index=False)

# Step 2 — assign strain pairs + build the strain Echo script

For every destination plate/well from Step 1 we assign a random **pair of strains** (the pairwise-interaction condition) or, for controls, a single strain by itself.

**Source plate:** a 384-well strain layout.

**Constraints (90 strains → >3741 unordered pairs, 9120 wells over 30 plates):**
- Every pair gets **≤ 3** replicates.
- Plates **1–19** are filled with a balanced random pass (each pair 1–2×); plates **20–30** then top every under-covered pair up to **≥ 2** replicates and carry the extra 3rd reps — so "starting at plate 20, every pair has at least 2".
- **≥ 1 monoculture control** per strain (100 nL strain + 100 nL blank from source well `A1`), scattered across all 30 plates.

Resulting allocation: 2190 pairs ×2 + 1551 pairs ×3 + 87 monocultures = 9120 wells.

In [9]:
strainseed = randseed_date + 1
rng = np.random.default_rng(strainseed)

strains = list(pick_list)
assume_blank = list(no_growth_strains)
N = len(strains)

pairs = list(itertools.combinations(range(N), 2))   # unordered strain pairs (by index)
P = len(pairs)
total = len(df_layout)
print(f"{P} unordered pairs, {total} destination wells")

# content[i] = ('mono', s) or ('pair', s1, s2) for destination-well row i of df_layout
content = [None] * total

# --- 1) monoculture controls: 1 per strain, scattered across all plates ---
mono_wells = rng.choice(total, size=N, replace=False)
for w_i, s_i in zip(mono_wells, rng.permutation(N)):
    content[w_i] = ('mono', strains[s_i])
mono_set = set(mono_wells.tolist())

# --- 2) split the remaining wells into early (plates 1-19) and late (plates 20+) ---
plate_of = df_layout['dest_plate'].values
rem   = [i for i in range(total) if i not in mono_set]
early = [i for i in rem if plate_of[i] <= 19]
late  = [i for i in rem if plate_of[i] >= 20]
n_early, n_late = len(early), len(late)
assert n_early >= P and (n_early - P) <= P, "early region cannot hold a balanced 1-2x pass"

# phase 1 (plates 1-19): every pair 1x, then a random subset gets a 2nd rep to fill the region
reps = np.ones(P, dtype=int)
second = rng.choice(P, size=n_early - P, replace=False)
reps[second] += 1
early_inst = list(range(P)) + list(second)
rng.shuffle(early_inst)
for w_i, p in zip(early, early_inst):
    a, b = pairs[p]
    content[w_i] = ('pair', strains[a], strains[b])

# phase 2 (plates 20+): complete every under-covered pair to >=2, then add 3rd reps (cap 3)
cur = reps.copy()
completion = [p for p in range(P) if cur[p] < 2]   # pairs still at 1 rep
for p in completion:
    cur[p] += 1
n_third = n_late - len(completion)
elig3 = [p for p in range(P) if cur[p] == 2]
third = rng.choice(elig3, size=n_third, replace=False)
late_inst = completion + list(third)
rng.shuffle(late_inst)
for w_i, p in zip(late, late_inst):
    a, b = pairs[p]
    content[w_i] = ('pair', strains[a], strains[b])
print(f"plates 1-19: {n_early} pair wells | plates 20+: {len(completion)} min-2 completions + {n_third} third reps")

# --- write strain assignment back onto df_layout ---
assert all(c is not None for c in content), "unassigned wells remain"
df_layout['well_type'] = [c[0] for c in content]
df_layout['strain1']   = [c[1] for c in content]
df_layout['strain2']   = [c[2] if c[0] == 'pair' else np.nan for c in content]
display(df_layout[['dest_plate', 'dest_well', 'well_type', 'strain1', 'strain2']])

#fill the strain2 column of the mono rows with a random selection from no_growth_strains
blank_well = rng.choice(assume_blank)
df_layout.loc[df_layout['well_type'] == 'mono', 'strain2'] = rng.choice(assume_blank, size=(df_layout['well_type'] == 'mono').sum(), replace=True)

#if assume_blank = {None}, then set strain2 to strain1
if assume_blank == [None]:
    df_layout.loc[df_layout['well_type'] == 'mono', 'strain2'] = df_layout.loc[df_layout['well_type'] == 'mono', 'strain1']
display(df_layout[df_layout['well_type'] == 'mono'])


3741 unordered pairs, 9120 destination wells
plates 1-19: 5797 pair wells | plates 20+: 1685 min-2 completions + 1551 third reps


,dest_plate,dest_well,well_type,strain1,strain2
0,1,B2,pair,C3,A19
1,1,B3,pair,P16,F16
2,1,B4,pair,N24,P11
3,1,B5,pair,K3,L11
4,1,B6,pair,N3,N15
...,...,...,...,...,...
9115,30,J9,pair,D20,F13
9116,30,J10,pair,F19,P17
9117,30,J11,pair,N19,K8
9118,30,J12,pair,N23,N15


,dest_plate,dest_row,dest_col,dest_well,fwd_idx,rev_idx,fwd_source_well,rev_source_well,well_type,strain1,strain2
296,1,O,12,O12,44,32,G17,F18,mono,P11,P11
312,2,B,6,B6,52,82,I9,N22,mono,N23,N23
409,2,F,15,F15,64,39,K9,H8,mono,C8,C8
631,3,B,17,B17,52,72,I9,N2,mono,D13,D13
639,3,C,3,C3,51,13,I7,D4,mono,P23,P23
...,...,...,...,...,...,...,...,...,...,...,...
8589,28,N,11,N11,17,11,C11,B24,mono,A5,A5
8870,29,M,6,M6,2,68,A5,L18,mono,D6,D6
8973,30,C,21,C21,85,76,O3,N10,mono,M15,M15
9002,30,E,6,E6,54,86,I13,P6,mono,C15,C15


In [10]:
# --- Strain Echo transfer df: two 100 nL transfers per destination well ---
strain_vol = 100  # nL per strain transfer
# strain_vol_div = 4
strain_vol_div = strain_vol // 25  # 25 nL per transfer (4 transfers per strain)    
strain_vol_div = False

strain_rows = []
for r in df_layout.itertuples():
    if r.well_type == 'pair':
        src_a, src_b = r.strain1, r.strain2
    else:  # monoculture control: strain + blank/media
        #src_a, src_b = r.strain1, blank_well
        src_a, src_b = r.strain1, r.strain2
    for src in (src_a, src_b):
        if strain_vol_div:
            for i in range(strain_vol_div):  # 4 transfers per strain to reach 100 nL total
                strain_rows.append({'Source Well': src, 'Destination Plate Name': r.dest_plate,
                                'Destination Well': r.dest_well, 'Transfer Volume': 25})
        else:
            strain_rows.append({'Source Well': src, 'Destination Plate Name': r.dest_plate,
                                'Destination Well': r.dest_well, 'Transfer Volume': strain_vol})
df_strain_echo = pd.DataFrame(strain_rows)

# --- sanity checks ---
pair_reps = Counter()
mono_counts = Counter()
for c in content:
    if c[0] == 'pair':
        pair_reps[frozenset((c[1], c[2]))] += 1
    else:
        mono_counts[c[1]] += 1
rc = np.array(list(pair_reps.values()))
assert len(pair_reps) == P, "not every pair is present"
assert rc.min() >= 2 and rc.max() <= 3, "replicate bounds violated"
assert all(mono_counts[s] >= 1 for s in strains), "a strain is missing its monoculture control"
if strain_vol_div:
    assert len(df_strain_echo) == 8 * total
else:
    assert len(df_strain_echo) == 2 * total
print(f"pairs: {len(pair_reps)}/{P} | reps min={rc.min()} max={rc.max()} | "
      f"2x={(rc==2).sum()} 3x={(rc==3).sum()}")
print(f"monoculture controls: {len(mono_counts)} strains, all >=1 rep")
print(f"strain echo rows: {len(df_strain_echo)}")

# --- save Echo-ready files (full + A/B halves for a refill break, mirroring the primer step) ---
df_layout.to_csv(f"strain_layout_{randseed_date}.csv", index=False)
df_strain_echo.to_csv(f"echo_strains_{randseed_date}.csv", index=False)
half = len(df_strain_echo) // 2
df_strain_echo.iloc[:half].to_csv(f"echo_strains_A_{randseed_date}.csv", index=False)
df_strain_echo.iloc[half:].to_csv(f"echo_strains_B_{randseed_date}.csv", index=False)
display(df_strain_echo)

pairs: 3741/3741 | reps min=2 max=3 | 2x=2190 3x=1551
monoculture controls: 87 strains, all >=1 rep
strain echo rows: 18240


,Source Well,Destination Plate Name,Destination Well,Transfer Volume
0,C3,1,B2,100
1,A19,1,B2,100
2,P16,1,B3,100
3,F16,1,B3,100
4,N24,1,B4,100
...,...,...,...,...
18235,K8,30,J11,100
18236,N23,30,J12,100
18237,N15,30,J12,100
18238,J18,30,J13,100


In [11]:
#sum how much will be shot from each well with a groupby on df_strain_echo
strain_shots = df_strain_echo.groupby('Source Well')['Transfer Volume'].sum()
display(strain_shots)

Source Well
A11    21400
A19    20900
A20    21600
A5     21500
A6     21400
       ...  
P16    20700
P17    21400
P20    21000
P23    21300
P8     21100
Name: Transfer Volume, Length: 87, dtype: int64

## section 3 - map back to the primer barcodes for minibar

create a minibar-ready primers csv with a SampleID containing the plate number and well number, a FwIndex, a FwPrimer, a RvIndex, and a RwPrimer

( using /home/rl/scripts/karl/drug_experiments/260303_8comm_retry/01_setup/20260312_echo_primers.ipynb
as reference )

In [12]:
primer_specs_path = './../../all_experiments/PrimerPlateSpecs.csv'

df_barcodes = pd.read_csv(
    primer_specs_path,
    usecols=['Well Position', 'Sequence Name', 'Sequence']
).copy()

# match the storage-well order used in the reference notebook
df_barcodes['Sequence'] = df_barcodes['Sequence'].str.replace(' ', '', regex=False)
df_barcodes['Storage'] = np.concatenate((rprimers_full, fprimers_full))[:len(df_barcodes)]

# forward barcode lookup
df_fw = (
    df_layout[['dest_plate', 'dest_well', 'fwd_source_well']]
    .merge(
        df_barcodes[['Storage', 'Sequence']],
        left_on='fwd_source_well',
        right_on='Storage',
        how='left',
        validate='m:1'
    )
    .rename(columns={'Sequence': 'FwIndex'})
    .drop(columns=['Storage', 'fwd_source_well'])
)

# reverse barcode lookup
df_rv = (
    df_layout[['dest_plate', 'dest_well', 'rev_source_well']]
    .merge(
        df_barcodes[['Storage', 'Sequence']],
        left_on='rev_source_well',
        right_on='Storage',
        how='left',
        validate='m:1'
    )
    .rename(columns={'Sequence': 'RvIndex'})
    .drop(columns=['Storage', 'rev_source_well'])
)

# combine into minibar-ready output
df_minibar = (
    df_fw.merge(df_rv, on=['dest_plate', 'dest_well'], how='inner')
)

df_minibar['SampleID'] = df_minibar.apply(
    lambda row: f"Plate{int(row['dest_plate']):02d}_{row['dest_well']}",
    axis=1
)

df_minibar['FwPrimer'] = 'AGRGTTYGATYMTGGCTCAG'
df_minibar['RwPrimer'] = 'CGGYTACCTTGTTACGACTT'

df_minibar = df_minibar[
    ['SampleID', 'FwIndex', 'FwPrimer', 'RvIndex', 'RwPrimer']
].copy()

assert df_minibar['FwIndex'].notna().all(), 'missing forward index assignments'
assert df_minibar['RvIndex'].notna().all(), 'missing reverse index assignments'

display(df_minibar)

out_tsv = f"minibar_primers_{randseed_date}.tsv"
df_minibar.to_csv(out_tsv, sep='\t', index=False)
print(f"saved {out_tsv}")

,SampleID,FwIndex,FwPrimer,RvIndex,RwPrimer
0,Plate01_B2,ATCGCCTACCGTGACTCAATCAAGAAGGGAAAGCAAGGTAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACAGTTTCCATCACTTCAGACTTGGGCGGYTAC...,CGGYTACCTTGTTACGACTT
1,Plate01_B3,ATCGCCTACCGTGACGCATAGTTCTGCATGATGGGTTAGAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACGAGTCTTGTGTCCCAGTTACCAGGCGGYTAC...,CGGYTACCTTGTTACGACTT
2,Plate01_B4,ATCGCCTACCGTGACGTGCAACTTTCCCACAGGTAGTTCAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACCACCCACACTTACTTCAGGACGTACGGYTAC...,CGGYTACCTTGTTACGACTT
3,Plate01_B5,ATCGCCTACCGTGACTGAAACCTAAGAAGGCACCGTATCAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACGCTGTGTTCCACTTCATTCTCCTGCGGYTAC...,CGGYTACCTTGTTACGACTT
4,Plate01_B6,ATCGCCTACCGTGACATGTCCCAGTTAGAGGAGGAAACAAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACGGAGTTCGTCCAGAGAAGTACACGCGGYTAC...,CGGYTACCTTGTTACGACTT
...,...,...,...,...,...
9115,Plate30_J9,ATCGCCTACCGTGACCTTGTCCAGGGTTTGTGTAACCTTAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACGCTAGGTCAATCTCCTTCGGAAGTCGGYTAC...,CGGYTACCTTGTTACGACTT
9116,Plate30_J10,ATCGCCTACCGTGACAGGTGATCCCAACAAGCGTAAGTAAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACAAGCGTTGAAACCTTTGTCCTCTCCGGYTAC...,CGGYTACCTTGTTACGACTT
9117,Plate30_J11,ATCGCCTACCGTGACAACGAGTCTCTTGGGACCCATAGAAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACAGAACGACTTCCATACTCGTGTGACGGYTAC...,CGGYTACCTTGTTACGACTT
9118,Plate30_J12,ATCGCCTACCGTGACAGAGGGTACTATGTGCCTCAGCACAGRGTTY...,AGRGTTYGATYMTGGCTCAG,ATCGCCTACCGTGACGAGTCTTGTGTCCCAGTTACCAGGCGGYTAC...,CGGYTACCTTGTTACGACTT


saved minibar_primers_20260721.tsv


## correct an error made during the experiment

create and save a new version of the strain_layout dataframe that switches the strains assigned to plate 1 with the strains assigned to plate B

In [14]:
# Create a corrected copy of the strain layout by swapping the strain assignments
# between plate 1 and plate B (plate label "B" or "plateb")
def normalize_plate_label(x):
    if pd.isna(x):
        return None
    return str(x).strip().lower().replace("plate", "", 1)

plate1_mask = df_layout["dest_plate"].map(normalize_plate_label) == "1"
plateB_mask = df_layout["dest_plate"].map(normalize_plate_label) == "2"

if plate1_mask.sum() == 0 or plateB_mask.sum() == 0:
    raise ValueError("Could not find both plate 1 and plate B in dest_plate.")
if plate1_mask.sum() != plateB_mask.sum():
    raise ValueError("Plate 1 and plate B do not have the same number of wells; cannot swap safely.")

swap_cols = ["well_type", "strain1", "strain2"]

df_layout_corrected = df_layout.copy()
plate1_rows = df_layout.loc[plate1_mask, swap_cols].copy()
plateB_rows = df_layout.loc[plateB_mask, swap_cols].copy()

df_layout_corrected.loc[plate1_mask, swap_cols] = plateB_rows.to_numpy()
df_layout_corrected.loc[plateB_mask, swap_cols] = plate1_rows.to_numpy()

out_csv = f"strain_layout_{randseed_date}_plate1_plateB_swapped.csv"
df_layout_corrected.to_csv(out_csv, index=False)

print(f"Saved corrected strain layout to {out_csv}")
display(
    df_layout_corrected[
        df_layout_corrected["dest_plate"].map(normalize_plate_label).isin(["1", "b"])
    ][["dest_plate", "dest_well", "well_type", "strain1", "strain2"]]
)

Saved corrected strain layout to strain_layout_20260721_plate1_plateB_swapped.csv


,dest_plate,dest_well,well_type,strain1,strain2
0,1,B2,pair,N23,C19
1,1,B3,pair,L24,J2
2,1,B4,pair,N21,J22
3,1,B5,pair,F17,K8
4,1,B6,mono,N23,N23
...,...,...,...,...,...
303,1,O19,pair,N19,F17
304,1,O20,pair,F19,D11
305,1,O21,pair,K3,C12
306,1,O22,pair,A5,P17
